# Очистка данных: Spend

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import help_130625_dam as h

DATA_PATH = os.path.join('..', 'Sources', 'Spend (Done).xlsx')
OUT_PATH  = os.path.join('..', 'data', 'cleaned', 'spend_clean.pkl')

## Загрузка и первичный осмотр

In [2]:
df = pd.read_excel(DATA_PATH)

# Переименование столбцов в snake_case
df.columns = [col.lower().replace(' ', '_') for col in df.columns]

print(f'Форма: {df.shape}')
print(f'Столбцы: {list(df.columns)}')
df.head()

Форма: (20779, 8)
Столбцы: ['date', 'source', 'campaign', 'impressions', 'spend', 'clicks', 'adgroup', 'ad']


,date,source,campaign,impressions,spend,clicks,adgroup,ad
0,2023-07-03,Google Ads,gen_analyst_DE,6,0.00,0,NaN,NaN
1,2023-07-03,Google Ads,performancemax_eng_DE,4,0.01,1,NaN,NaN
2,2023-07-03,Facebook Ads,NaN,0,0.00,0,NaN,NaN
3,2023-07-03,Google Ads,NaN,0,0.00,0,NaN,NaN
4,2023-07-03,CRM,NaN,0,0.00,0,NaN,NaN


In [3]:
h.descr_df(df, include='all', show_sample_rows=True)

,Название признака,Тип данных,Количество значений,Пропуски (NaN),Уникальных значений,Пример строка 1,Пример строка 2,Пример строка 3,Минимум,Среднее,Медиана,Максимум
0,date,datetime64[us],20779,0,355,2023-07-03 00:00:00,2023-07-03 00:00:00,2023-07-03 00:00:00,NaN,NaN,NaN,NaN
1,source,str,20779,0,14,Google Ads,Google Ads,Facebook Ads,NaN,NaN,NaN,NaN
2,campaign,str,14785,5994,51,gen_analyst_DE,performancemax_eng_DE,NaN,NaN,NaN,NaN,NaN
3,impressions,int64,20779,0,4003,6,4,0,0.0,2458.203475,63.00,431445.0
4,spend,float64,20779,0,2859,0.0,0.01,0.0,0.0,7.195892,0.58,774.0
5,clicks,int64,20779,0,552,0,1,0,0.0,23.990616,1.00,2415.0
6,adgroup,str,13951,6828,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ad,str,13951,6828,176,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Пропущенные значения
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
display(
    pd.DataFrame({'Пропуски': missing, '% пропусков': missing_pct})
    .query('Пропуски > 0')
)
print('Строк без пропусков:', df.dropna().shape[0])

,Пропуски,% пропусков
campaign,5994,28.85
adgroup,6828,32.86
ad,6828,32.86


Строк без пропусков: 13874


In [5]:
df['date'].unique()[:10]

<DatetimeArray>
['2023-07-03 00:00:00', '2023-07-04 00:00:00', '2023-07-05 00:00:00',
 '2023-07-06 00:00:00', '2023-07-07 00:00:00', '2023-07-08 00:00:00',
 '2023-07-09 00:00:00', '2023-07-10 00:00:00', '2023-07-11 00:00:00',
 '2023-07-12 00:00:00']
Length: 10, dtype: datetime64[us]

In [6]:
n_before = len(df)

full_dupes = df.duplicated().sum()
print(f'Полных дубликатов: {full_dupes}')

# У Spend нет Id — дубликат это полное совпадение всех полей
if full_dupes > 0:
    print('Пример дубликатов:')
    display(df[df.duplicated(keep=False)].sort_values('date').head(10))
    df = df.drop_duplicates().reset_index(drop=True)

print(f'Строк до: {n_before}  →  после: {len(df)}  (удалено: {n_before - len(df)})')


Полных дубликатов: 917
Пример дубликатов:


,date,source,campaign,impressions,spend,clicks,adgroup,ad
753,2023-07-23,Bloggers,NaN,0,0.0,0,NaN,NaN
755,2023-07-23,Bloggers,NaN,0,0.0,0,NaN,NaN
768,2023-07-24,Bloggers,NaN,0,0.0,0,NaN,NaN
789,2023-07-24,Bloggers,NaN,0,0.0,0,NaN,NaN
841,2023-07-25,Bloggers,NaN,0,0.0,0,NaN,NaN
844,2023-07-25,Bloggers,NaN,0,0.0,0,NaN,NaN
895,2023-07-26,Bloggers,NaN,0,0.0,0,NaN,NaN
899,2023-07-26,Bloggers,NaN,0,0.0,0,NaN,NaN
950,2023-07-27,Bloggers,NaN,0,0.0,0,NaN,NaN
958,2023-07-27,Bloggers,NaN,0,0.0,0,NaN,NaN


Строк до: 20779  →  после: 19862  (удалено: 917)


## Типы данных: дата

In [7]:
DATE_COLS = ['date']

for col in DATE_COLS:
    # Явно указываем формат YYYY-MM-DD
    df[col] = pd.to_datetime(df[col], format='%Y-%m-%d', errors='coerce')

# Проверяем результат
print('Типы после парсинга:')
print(df[DATE_COLS].dtypes)
print()

# Считаем NaT (не распарсились)
for col in DATE_COLS:
    nat_count = df[col].isna().sum()
    print(f'{col}: NaT = {nat_count} ({nat_count/len(df)*100:.2f}%)')

Типы после парсинга:
date    datetime64[us]
dtype: object

date: NaT = 0 (0.00%)


## Числовые поля

In [8]:
# Проверяем наличие "грязных" значений в числовых полях
NUM_COLS = ['impressions', 'spend', 'clicks']

for col in NUM_COLS:
    print(f'--- {col} ---')
    print(f'  Тип: {df[col].dtype}')
    print(f'  Мин: {df[col].min()}, Макс: {df[col].max()}')
    neg = (df[col] < 0).sum()
    print(f'  Отрицательных значений: {neg}')


--- impressions ---
  Тип: int64
  Мин: 0, Макс: 431445
  Отрицательных значений: 0
--- spend ---
  Тип: float64
  Мин: 0.0, Макс: 774.0
  Отрицательных значений: 0
--- clicks ---
  Тип: int64
  Мин: 0, Макс: 2415
  Отрицательных значений: 0


In [9]:
# # Если spend, impressions, clicks пришли как строки — очищаем
# def clean_numeric(series):
#     """Убирает пробелы, знаки валюты, переносы строк; приводит к float."""
#     if series.dtype == object:
#         series = (
#             series.astype(str)
#             .str.replace(r'[\s€$£\xa0\n\r]', '', regex=True)
#             .str.replace(',', '.', regex=False)
#         )
#         return pd.to_numeric(series, errors='coerce')
#     return series

# for col in NUM_COLS:
#     df[col] = clean_numeric(df[col])

# print('Типы после очистки:')
# print(df[NUM_COLS].dtypes)


## Пропущенные значения

Этот датасет это единственный источник значений campaign, adgroup, ad, поэтому дозаполнить их не 
получиться. Пропущенные значения заменяем на Unknown и меняем тип на category

In [10]:
# campaign, adgroup, ad — ~30% пропусков, заполняем 'Unknown'
FILL_UNKNOWN = ['campaign', 'adgroup', 'ad']

for col in FILL_UNKNOWN:
    n_miss = df[col].isna().sum()
    df[col] = df[col].fillna('Unknown')
    print(f'{col}: заполнено {n_miss} пропусков → "Unknown"')

# Проверка
print('\nПропуски после заполнения:')
missing_after = df.isnull().sum()
display(
    pd.DataFrame({'Пропуски': missing_after, '% пропусков': (missing_after / len(df) * 100).round(2)})
    .query('Пропуски > 0')
)

CAT_COLS = ['source', 'campaign', 'adgroup', 'ad']

for col in CAT_COLS:
    # Нормализация: убираем лишние пробелы по краям
    df[col] = df[col].str.strip()
    df[col] = df[col].astype('category')
    print(f'{col}: {df[col].nunique()} уникальных значений')

campaign: заполнено 5077 пропусков → "Unknown"
adgroup: заполнено 5911 пропусков → "Unknown"
ad: заполнено 5911 пропусков → "Unknown"

Пропуски после заполнения:


,Пропуски,% пропусков


source: 14 уникальных значений
campaign: 52 уникальных значений
adgroup: 25 уникальных значений
ad: 177 уникальных значений


## Категориальные поля

In [11]:
# уникальные источники и кол-во записей
print('Источники (source):')
print(df['source'].value_counts())
print(f'\nУникальных источников: {df["source"].nunique()}')


Источники (source):
source
Facebook Ads      9569
Tiktok Ads        2985
Youtube Ads       1784
Google Ads        1266
Telegram posts     836
Webinar            766
Bloggers           632
SMM                571
Organic            514
CRM                355
Test               262
Partnership        234
Offline             61
Radio               27
Name: count, dtype: int64

Уникальных источников: 14


## Итоги

In [12]:
print(f'Итоговая форма датасета: {df.shape}')
print()
print('Типы данных:')
print(df.dtypes)
print()
df.head()

Итоговая форма датасета: (19862, 8)

Типы данных:
date           datetime64[us]
source               category
campaign             category
impressions             int64
spend                 float64
clicks                  int64
adgroup              category
ad                   category
dtype: object



,date,source,campaign,impressions,spend,clicks,adgroup,ad
0,2023-07-03,Google Ads,gen_analyst_DE,6,0.00,0,Unknown,Unknown
1,2023-07-03,Google Ads,performancemax_eng_DE,4,0.01,1,Unknown,Unknown
2,2023-07-03,Facebook Ads,Unknown,0,0.00,0,Unknown,Unknown
3,2023-07-03,Google Ads,Unknown,0,0.00,0,Unknown,Unknown
4,2023-07-03,CRM,Unknown,0,0.00,0,Unknown,Unknown


## Сохранение

In [13]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
df.to_pickle(OUT_PATH)

# Итоги
summary_data = {
    'Метрика': [
        'Строк исходно',
        'Строк после очистки',
        'Удалено дубликатов',
        'Диапазон дат',
        'Каналов (source)',
        'Итого Spend, €',
        'Пропуски после заполнения'
    ],
    'Значение': [
        n_before,
        len(df),
        n_before - len(df),
        f'{df["date"].min().date()} → {df["date"].max().date()}',
        df['source'].nunique(),
        f'{df["spend"].sum():,.2f}',
        df.isnull().sum().sum()
    ]
}

print(f'Сохранено: {OUT_PATH}')
display(pd.DataFrame(summary_data))


Сохранено: ..\data\cleaned\spend_clean.pkl


,Метрика,Значение
0,Строк исходно,20779
1,Строк после очистки,19862
2,Удалено дубликатов,917
3,Диапазон дат,2023-07-03 → 2024-06-21
4,Каналов (source),14
5,"Итого Spend, €","149,523.45"
6,Пропуски после заполнения,0


## Описание датасета

**Источник:** `Spend.xlsx` — данные рекламных кабинетов  
**Назначение:** учет маркетинговых затрат для расчета ROI/ROAS и анализа эффективности каналов привлечения

| Столбец | Тип | Описание |
|---|---|---|
| `date` | `datetime` | Дата расхода (приведена к формату YYYY-MM-DD) |
| `source` | `category` | Рекламный канал/источник (Google, Facebook и др.) |
| `campaign` | `category` | Название рекламной кампании |
| `impressions` | `int64` | Количество показов объявлений |
| `spend` | `float64` | Сумма фактических затрат в валюте кабинета |
| `clicks` | `int64` | Количество переходов (кликов) |
| `adgroup` | `category` | Группа объявлений |
| `ad` | `category` | Конкретное объявление/креатив |

**Объём:** 19 862 записей (после очистки), 8 столбцов  
**Ключевые связи:**
- `date` + `source` → агрегация для сопоставления с выручкой в `05_data_merging`
- `campaign` → декомпозиция затрат до уровня оффера/гео
- `spend` / `clicks` → расчет CPC (Cost Per Click)